In [1]:
import os
import pandas as pd
from langchain_google_genai import ChatGoogleGenerativeAI
import json

d:\nerii\Desktop\Work\DVS\Manson Maze\py\env\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


In [14]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [15]:
from utils.navigation import Agent, Maze, explore_maze

# Settings

In [16]:
test = "gemini-3-flash-preview"
iterations = 2


output_dir = os.path.join("outputs", test)
os.makedirs(output_dir, exist_ok=True)

# Run Exploration

In [17]:
model = ChatGoogleGenerativeAI(model=test, request_timeout=180, max_retries=3)

params = ["travel_logs", "decision_logs", "analysis_logs", "last_notes", "end_causes"]

data = {}
for key in params:
    file_path = os.path.join(output_dir, f"{key}.json")
    
    if os.path.exists(file_path):
        with open(file_path, 'r') as f:
            data[key] = json.load(f)
    else:
        data[key] = []

data

{'travel_logs': [],
 'decision_logs': [],
 'analysis_logs': [],
 'last_notes': [],
 'end_causes': []}

In [ ]:
initial_iteration = len(data["travel_logs"]) 
last_iteration = initial_iteration + iterations


for iter in range(initial_iteration, last_iteration):
        
    maze = Maze()
    agent = Agent(model=model)

    # Inject past notes
    if len(data["last_notes"]) > 0:
        agent.last_notes = "\n".join(f"Attempt {note['iteration']}: {note['data']}" for note in data["last_notes"])

    # Run Iteration
    new_data, agent = explore_maze(agent, maze)

    print("Causes of failure:", agent.status, "/ note:", new_data["last_notes"])

    # Store outputs
    for key, new_items in new_data.items():
        if key in data:
            data[key].append({"iteration": iter, "data": new_items})
        else:
            data[key] = [{"iteration": iter, "data": new_items}]


Starting exploration
AI Ready: True


In [ ]:
for key, content in data.items():
    filename = os.path.join(output_dir, f"{key}.json")
    
    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(content, f, indent=4)

outputs\gemini-3-flash-preview\travel_logs.json
outputs\gemini-3-flash-preview\decision_logs.json
outputs\gemini-3-flash-preview\analysis_logs.json
outputs\gemini-3-flash-preview\past_notes.json
outputs\gemini-3-flash-preview\end_causes.json
